# TransferBench

## A Benchmarking Framework for Transfer Learning

TransferBench is a benchmarking framework designed to evaluate and compare transfer learning strategies across multiple deep learning architectures for image classification.

This notebook demonstrates the complete workflow from data loading to benchmarking.

### Workflow

- Environment Setup
- Dataset Preparation
- Model Configuration
- Training
- Evaluation
- Visualization
- Benchmark Report

In [52]:
import os
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torchvision

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch :", torch.__version__)
print("TorchVision :", torchvision.__version__)
print("Device :", device)

PyTorch : 2.12.1+cu132
TorchVision : 0.27.1+cu132
Device : cuda


In [64]:
print("=" * 60)
print("TransferBench Environment Ready")
print("=" * 60)

TransferBench Environment Ready


In [53]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random Seed Fixed :", SEED)

Random Seed Fixed : 42


# Dataset Preparation

In this section we download the dataset, apply preprocessing transforms, and create the training and validation dataloaders.

In [65]:
print(f"Classes: {train_dataset.classes}")
print(f"Training Batches: {len(train_loader)}")
print(f"Validation Batches: {len(test_loader)}")

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Training Batches: 1563
Validation Batches: 313


In [55]:
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=test_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Train Samples :", len(train_dataset))
print("Test Samples :", len(test_dataset))

Train Samples : 50000
Test Samples : 10000


In [66]:
print("Dataset visualization saved successfully.")

Dataset visualization saved successfully.


In [57]:
classes = train_dataset.classes

images, labels = next(iter(train_loader))

plt.figure(figsize=(12,6))

for i in range(8):
    plt.subplot(2,4,i+1)
    plt.imshow(images[i].permute(1,2,0))
    plt.title(classes[labels[i]])
    plt.axis("off")

plt.tight_layout()

os.makedirs("figures", exist_ok=True)

plt.savefig("figures/dataset_samples.png", dpi=300)

plt.show()

# Model Initialization

In this section we initialize the pretrained deep learning model.

For the first benchmark experiment we use **ResNet18** pretrained on ImageNet.

Only the classifier layer will be trained during the first experiment (Freeze Feature Extractor).

In [59]:
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)

NUM_CLASSES = len(train_dataset.classes)

model.fc = nn.Linear(
    model.fc.in_features,
    NUM_CLASSES
)

model = model.to(device)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [60]:
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable Parameters : {trainable:,}")
print(f"Total Parameters     : {total:,}")

Trainable Parameters : 5,130
Total Parameters     : 11,181,642


# Optimizer & Loss Function

Only the classification head is optimized.

Loss Function:

- CrossEntropyLoss

Optimizer:

- Adam

Learning Rate:

- 0.001

In [62]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.fc.parameters(),
    lr=1e-3
)

print(criterion)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [63]:
EPOCHS = 5

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

print("Training configuration ready.")
print(f"Epochs      : {EPOCHS}")
print(f"Batch Size  : {BATCH_SIZE}")
print(f"Device      : {device}")

Training configuration ready.
Epochs      : 5
Batch Size  : 32
Device      : cuda


# Model Initialization

In this experiment, we use **ResNet18** pretrained on ImageNet as the baseline model.

To evaluate the effectiveness of transfer learning, only the final classification layer is trainable while the backbone remains frozen.

This strategy significantly reduces training time and minimizes overfitting on small datasets.

In [67]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

NUM_CLASSES = len(train_dataset.classes)

weights = ResNet18_Weights.DEFAULT

model = resnet18(weights=weights)

# Freeze Backbone
for param in model.parameters():
    param.requires_grad = False

# Replace Classification Head
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, NUM_CLASSES)
)

# Train only classifier
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to(device)

print("=" * 60)
print(model.__class__.__name__)
print("=" * 60)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable Parameters : {trainable:,}")
print(f"Total Parameters     : {total:,}")
print(f"Frozen Parameters    : {total-trainable:,}")

ResNet
Trainable Parameters : 5,130
Total Parameters     : 11,181,642
Frozen Parameters    : 11,176,512


In [68]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

print("=" * 60)
print("Optimizer Ready")
print("=" * 60)

print(optimizer)

Optimizer Ready
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0001
)


# Training Configuration

The following hyperparameters are used during the first benchmark experiment.

- Model: ResNet18
- Strategy: Freeze Feature Extractor
- Dataset: CIFAR-10
- Optimizer: Adam
- Loss Function: CrossEntropyLoss

In [70]:
EPOCHS = 5

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_accuracy = 0.0

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

print("=" * 60)
print("Training Configuration")
print("=" * 60)

print(f"Epochs        : {EPOCHS}")
print(f"Batch Size    : {BATCH_SIZE}")
print(f"Learning Rate : {optimizer.param_groups[0]['lr']}")
print(f"Device        : {device}")

Training Configuration
Epochs        : 5
Batch Size    : 32
Learning Rate : 0.001
Device        : cuda
